# 05 · 构建数据集：合并 X 和 Y，划分 train / val / test（GreenPath）

**这一步做什么**：把前面几步产出的表合成一张可以直接训练的"xy 表"，过滤质量不合格的格子，按**空间区块**划分数据，并给这一版数据集登记版本。

| | 路径（相对数据根目录） |
|---|---|
| 输入：格子主表（含卫星图路径） | `processed/cells.parquet`（来自 01、02） |
| 输入：OSM / WorldCover 特征 | `processed/osm_features.parquet`（来自 03） |
| 输入：标签 Y | `processed/labels_cell.parquet`（来自 04） |
| 输出：**主表** | `datasets/{版本}/dataset.parquet` |
| 输出：按集合切好的视图 | `datasets/{版本}/train.parquet`、`val.parquet`、`test.parquet` |
| 输出：列分组说明 | `datasets/{版本}/schema.json` |
| 输出：来源与版本记录 | `datasets/{版本}/manifest.json` |
| 输出：数据集说明卡 | `datasets/{版本}/DATASET_CARD.md` |
| 输出：运行记录 | `runs/{时间戳}/05_build_dataset_manifest.json` |

**数据集版本是"冻结"的**：同一个版本号已经存在时，默认**拒绝覆盖**（改了数据或标签就升版本号 `v1` → `v2`），这样任何一个模型都能说清楚它是用哪一版数据训练的。

**划分方式（最关键）**：相邻格子几乎一样，随机划分会让测试分数虚高（空间泄漏）。所以
1. 先把地图切成 H3 **区块**（默认分辨率 8，约 0.7 km²）；
2. 以**区块为单位**分配到 train / val / test，同一区块的格子一定在同一份里；
3. 紧挨着测试区块的训练 / 验证格子丢掉一圈（**缓冲区**），防止区块边界上的泄漏；
4. 另外给 train+val 的区块分 5 折（`fold` 列），样本少时做分组交叉验证用；以后数据多了不用这一列就行；
5. 如果以后有第二座城市，把 `TEST_CITIES` 设成它，**整座城市当测试集**，这才是真正的跨城测试。

**测试集只在最终报告时看一次**，调参全部用 train / val 或交叉验证。

## 1. 安装依赖（只在 Colab 里会执行）

In [1]:
import sys, subprocess
if "google.colab" in sys.modules:      # 只有在 Colab 里才自动安装；本地环境请提前装好
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "h3>=4", "pandas", "pyarrow", "gcsfs", "matplotlib", "tqdm", "fsspec"], check=True)

## 2. 配置
- `DATASET_VERSION`：这一版数据集的名字。**改了任何输入或参数，就升版本号。**
- `BLOCK_RES`：区块大小（H3 分辨率）。8 ≈ 0.7 km²；越大（数字越小）泄漏越少，但区块数越少。
- `SPLIT_RATIOS`：train / val / test 的格子数比例（按区块分配，所以只是近似）。
- `BUFFER_RINGS`：测试区块周围丢几圈格子。`0` = 不丢。
- `N_FOLDS`：train+val 分几折。
- `MAX_YEAR_GAP`：街景拍摄年份（取该格子最早和最晚拍摄的中点）和卫星图年份相差超过这么多年的格子丢掉。
- `MIN_LABELED_DIMS`：至少有几个维度的标签不是缺失，格子才保留。
- `TEST_CITIES`：非空时，这些城市的格子整体当测试集（跨城测试），其余城市再按区块分 train / val。
- `OVERWRITE`：允许覆盖已存在的版本（一般不要开）。

In [ ]:
DATASET_VERSION = "v1"
BLOCK_RES = 8
SPLIT_RATIOS = (0.70, 0.15, 0.15)       # train, val, test
BUFFER_RINGS = 1
N_FOLDS = 5
MAX_YEAR_GAP = 3
MIN_LABELED_DIMS = 1
TEST_CITIES = []                        # 例如 ["providence"]
SEED = 42
OVERWRITE = True        # 如果为 True，则会覆盖已存在的数据

STORAGE = "local"                              # "local" | "gcs" | "drive"
GCS_ROOT = "gs://greenpath-data"
DRIVE_ROOT = "/content/drive/MyDrive/greenpath-data"
LOCAL_ROOT = "./data"

## 3. 连接存储，读入三张表

In [3]:
import os, io, json, datetime
import fsspec
import pandas as pd

STORAGE_OPTS = {}
if STORAGE == "gcs":
    try:
        from google.colab import auth
        auth.authenticate_user()                 # Colab 里登录；本地请先 gcloud auth application-default login
    except ImportError:
        pass
    ROOT = GCS_ROOT
    STORAGE_OPTS = {"token": "google_default"}
    fs = fsspec.filesystem("gcs", **STORAGE_OPTS)
elif STORAGE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DRIVE_ROOT
    fs = fsspec.filesystem("file")
else:
    ROOT = os.environ.get("GREENPATH_ROOT", LOCAL_ROOT)   # 以后容器里用环境变量指定数据目录
    fs = fsspec.filesystem("file")

def path(*parts):
    # 拼出数据根目录下的完整路径，例如 path("processed", "cells.parquet")
    return "/".join([ROOT.rstrip("/"), *parts])

def makedirs(p):
    if STORAGE != "gcs":            # GCS 没有真正的目录，不需要创建
        os.makedirs(p, exist_ok=True)

def read_parquet(p):
    return pd.read_parquet(p, storage_options=STORAGE_OPTS or None)

def write_parquet(df, p):
    df.to_parquet(p, index=False, storage_options=STORAGE_OPTS or None)

def write_manifest(manifest, name):
    run_ts = datetime.datetime.utcnow().strftime("%Y-%m-%dT%H-%M")
    makedirs(path("runs", run_ts))
    with fs.open(path("runs", run_ts, name), "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    return manifest

try:                                          # 图里的中文用系统自带的中文字体显示
    import matplotlib
    matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "Noto Sans CJK SC",
                                              "Microsoft YaHei", "SimHei", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
except ImportError:
    pass

now_iso = lambda: datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z"
print("数据根目录:", ROOT)

数据根目录: ./data


In [4]:
import numpy as np, hashlib, h3
import matplotlib.pyplot as plt

DIMS = ["traffic_intensity", "quietness", "shade", "naturalness",
        "scenic_appeal", "historic_character", "liveliness", "visual_openness"]
OUT_DIR = f"datasets/{DATASET_VERSION}"
if fs.exists(path(OUT_DIR, "manifest.json")) and not OVERWRITE:
    raise FileExistsError(f"数据集 {DATASET_VERSION} 已经存在，是冻结的版本。改了数据或参数请把 DATASET_VERSION 升一级（如 v2）；确实要覆盖才设 OVERWRITE = True。")

cells = read_parquet(path("processed", "cells.parquet"))
osm = read_parquet(path("processed", "osm_features.parquet"))
labels = read_parquet(path("processed", "labels_cell.parquet"))
with fs.open(path("processed", "labels_meta.json"), "r") as f:
    labels_meta = json.load(f)
print(f"格子 {len(cells)} · OSM 特征 {len(osm)} · 标签 {len(labels)}")
print(f"标签来自：{labels_meta['model']} · prompt {labels_meta['prompt_version']} · 估算花费 ${labels_meta['est_cost_usd']}")

FileExistsError: 数据集 v1 已经存在，是冻结的版本。改了数据或参数请把 DATASET_VERSION 升一级（如 v2）；确实要覆盖才设 OVERWRITE = True。

## 4. 合并
按 `h3_index` 把三张表横向拼起来。列分三类：
- **图像输入**：`tile_path` 等（卫星图本身不放进表里，只存相对路径）；
- **表格输入**：OSM / WorldCover 特征；
- **标签 Y**：8 个维度（0–1，缺失为 NaN）及其标准差。

In [ ]:
osm_cols = [c for c in osm.columns if c not in ("cell_id", "h3_index", "updated_at")]
cat_cols = [c for c in osm_cols if not pd.api.types.is_numeric_dtype(osm[c])]   # 如 wc_dominant（类别，不是数字）
tab_cols = [c for c in osm_cols if c not in cat_cols]
std_cols = [f"{d}_std" for d in DIMS]

df = (cells.drop(columns=["updated_at"], errors="ignore")
      .merge(osm.drop(columns=["cell_id", "updated_at"], errors="ignore"), on="h3_index", how="left", validate="1:1")
      .merge(labels[["h3_index", "n_usable", *DIMS, *std_cols, "prompt_version", "model"]]
             .rename(columns={"prompt_version": "label_prompt_version", "model": "label_model"}),
             on="h3_index", how="left", validate="1:1"))
print(f"合并后 {len(df)} 行，{df.shape[1]} 列；表格特征 {len(tab_cols)} 个，类别特征 {cat_cols}")

## 5. 过滤
每一步都打印"还剩多少个格子"。被丢掉的原因会写进数据集说明卡。

In [ ]:
funnel = [("合并后的格子", len(df))]
def step(name, mask):
    global df
    df = df[mask(df)].reset_index(drop=True)
    funnel.append((name, len(df)))

step("卫星图可用", lambda d: d.tile_ok.fillna(False).astype(bool))
step(f"至少 {MIN_LABELED_DIMS} 个维度有标签", lambda d: d[DIMS].notna().sum(axis=1) >= MIN_LABELED_DIMS)
yr = lambda s: pd.to_datetime(s, format="%Y-%m").dt.year
df["street_year_mid"] = (yr(df.capture_min) + yr(df.capture_max)) / 2
df["year_gap"] = (df.naip_year.astype(float) - df.street_year_mid).abs()
step(f"卫星图与街景年份相差 ≤ {MAX_YEAR_GAP} 年", lambda d: d.year_gap <= MAX_YEAR_GAP)
step("OSM 特征齐全", lambda d: d.road_total_m.notna())

for name, n in funnel:
    print(f"{name:<32}{n:>8,}")

## 6. 划分：区块 → train / val / test，缓冲区，交叉验证折
下面三个函数没有副作用，可以单独测试：
- `assign_splits`：按区块分配 train / val / test（贪心地凑够各自的格子数比例）；
- `apply_buffer`：丢掉紧挨着测试集的训练 / 验证格子；
- `assign_folds`：把 train+val 的区块分成 `N_FOLDS` 折，每折的格子数尽量均衡。

In [ ]:
def assign_splits(d, ratios, test_cities, seed):
    # 指定的测试城市：整座城市的格子直接进 test（按格子自己的城市判断）；其余格子按区块分配
    rng = np.random.default_rng(seed)
    is_test_city = d.city.isin(test_cities) if test_cities else pd.Series(False, index=d.index)
    blocks = d[~is_test_city].groupby("block_id").size().rename("n").reset_index()
    total = blocks.n.sum()
    if test_cities:
        test_target, val_target = 0, total * ratios[1] / (ratios[0] + ratios[1])
    else:
        test_target, val_target = total * ratios[2], total * ratios[1]
    split_of, acc_t, acc_v = {}, 0, 0
    for i in rng.permutation(len(blocks)):
        b, n = blocks.iloc[i].block_id, blocks.iloc[i].n
        if acc_t < test_target:
            split_of[b] = "test"; acc_t += n
        elif acc_v < val_target:
            split_of[b] = "val"; acc_v += n
        else:
            split_of[b] = "train"
    return d.block_id.map(split_of).where(~is_test_city, "test")

def apply_buffer(d, rings):
    # 丢掉距离任何测试格子 ≤ rings 圈的训练 / 验证格子；测试格子保持不动
    if rings <= 0:
        return d.index
    test_set = set(d.loc[d.split == "test", "h3_index"])
    drop = [i for i, r in d[d.split != "test"].iterrows()
            if any(n in test_set for n in h3.grid_disk(r.h3_index, rings))]
    return d.index.difference(drop)

def assign_folds(d, n_folds, seed):
    # 只对 train+val 的区块分折；贪心地把每个区块放进当前格子数最少的折
    rng = np.random.default_rng(seed + 1)
    b = d[d.split.isin(["train", "val"])].groupby("block_id").size()
    b = b.iloc[rng.permutation(len(b))].sort_values(ascending=False, kind="stable")
    load = np.zeros(n_folds, dtype=int); fold_of = {}
    for blk, n in b.items():
        k = int(load.argmin()); fold_of[blk] = k; load[k] += n
    return d.block_id.map(fold_of).astype("Int64")

In [ ]:
df["block_id"] = [h3.cell_to_parent(i, BLOCK_RES) for i in df.h3_index]
df["split"] = assign_splits(df, SPLIT_RATIOS, TEST_CITIES, SEED)
n_before = len(df)
df = df.loc[apply_buffer(df, BUFFER_RINGS)].reset_index(drop=True)
funnel.append((f"缓冲区：丢掉紧挨测试集的 {BUFFER_RINGS} 圈格子", len(df)))
df["fold"] = assign_folds(df, N_FOLDS, SEED)

# 检查：没有区块同时出现在两个集合里
leak = df.groupby("block_id").split.nunique()
assert (leak == 1).all(), "有区块同时属于多个集合！"
print(f"区块总数 {df.block_id.nunique()}；缓冲区丢掉 {n_before - len(df)} 个格子")

summary = df.groupby("split").agg(格子数=("h3_index", "size"), 区块数=("block_id", "nunique")).reindex(["train", "val", "test"]).fillna(0).astype(int)
summary["占比"] = (summary.格子数 / summary.格子数.sum()).round(2)
summary

## 7. 检查划分是否合理
- **地图**：三种颜色应该是成块分布，而不是零散混在一起；
- **标签均值**：各集合的均值不应该差太多，差得很大说明某个集合碰巧抽到了特殊区域；
- **缺失率**：每个维度的标签缺失比例。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for s, c in [("train", "tab:blue"), ("val", "tab:orange"), ("test", "tab:red")]:
    x = df[df.split == s]
    ax.scatter(x.lon, x.lat, s=14, c=c, label=f"{s} ({len(x)})", alpha=.85)
ax.set_aspect(1 / np.cos(np.radians(df.lat.mean()))); ax.legend(); ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.set_title(f"划分（区块分辨率 {BLOCK_RES}，缓冲 {BUFFER_RINGS} 圈）")
plt.show()

chk = df.groupby("split")[DIMS].mean().reindex(["train", "val", "test"]).round(2)
chk.loc["缺失率（全体）"] = df[DIMS].isna().mean().round(2)
chk

## 8. 写出数据集
- `dataset.parquet`：所有行，带 `split`、`block_id`、`fold` 列；
- `train / val / test.parquet`：按 `split` 切好的视图，训练脚本直接读；
- `schema.json`：列分组（标识、图像、表格特征、类别特征、标签、标签质量、来源），训练代码按它取列，不用硬编码；
- `manifest.json`：输入文件的哈希、prompt 版本和哈希、OSM 快照日期、过滤条件、划分参数。
数值列统一存成 float32。

In [ ]:
id_cols = ["cell_id", "h3_index", "city", "lat", "lon"]
split_cols = ["split", "block_id", "fold"]
image_cols = ["tile_path", "naip_year", "ground_m", "tile_px"]
quality_cols = ["n_images", "n_usable", "capture_min", "capture_max", "year_gap", *std_cols]
prov_cols = ["label_model", "label_prompt_version", "dataset_version"]
df["dataset_version"] = DATASET_VERSION

order = id_cols + split_cols + image_cols + tab_cols + cat_cols + DIMS + quality_cols + prov_cols
out = df[order].copy()
for c in tab_cols + DIMS + std_cols + ["year_gap"]:
    out[c] = out[c].astype("float32")
out["fold"] = out.fold.astype("Int64")
out = out.sort_values(["split", "cell_id"]).reset_index(drop=True)

makedirs(path(OUT_DIR))
write_parquet(out, path(OUT_DIR, "dataset.parquet"))
for s in ["train", "val", "test"]:
    write_parquet(out[out.split == s].reset_index(drop=True), path(OUT_DIR, f"{s}.parquet"))

schema = {"id": id_cols, "split": split_cols, "image": image_cols, "tabular": tab_cols, "categorical": cat_cols,
          "targets": DIMS, "target_std": std_cols, "quality": quality_cols, "provenance": prov_cols,
          "notes": "targets 取值 0–1，缺失为 NaN；tile_path 相对数据根目录；fold 仅 train/val 有值"}
with fs.open(path(OUT_DIR, "schema.json"), "w") as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)
print(f"已写出 {OUT_DIR}/：{len(out)} 行 × {out.shape[1]} 列")
out.groupby("split").size().reindex(["train", "val", "test"])

## 9. 版本记录和数据集说明卡

In [ ]:
def sha256(rel):
    h = hashlib.sha256()
    with fs.open(path(rel), "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]

osm_meta = {}
try:
    pbfs = [p for p in fs.ls(path("raw", "osm")) if str(p).endswith(".meta.json")]
    if pbfs:
        with fs.open(pbfs[0], "r") as f:
            osm_meta = json.load(f)
except Exception:
    pass

manifest = {
    "dataset_version": DATASET_VERSION, "created_at": now_iso(), "seed": SEED,
    "inputs": {rel: sha256(rel) for rel in ["processed/cells.parquet", "processed/osm_features.parquet", "processed/labels_cell.parquet"]},
    "labels": {k: labels_meta.get(k) for k in ["model", "prompt_version", "prompt_sha", "thinking_budget", "temperature", "images_scored", "est_cost_usd"]},
    "osm_snapshot": osm_meta,
    "params": {"block_res": BLOCK_RES, "split_ratios": SPLIT_RATIOS, "buffer_rings": BUFFER_RINGS, "n_folds": N_FOLDS,
               "max_year_gap": MAX_YEAR_GAP, "min_labeled_dims": MIN_LABELED_DIMS, "test_cities": TEST_CITIES},
    "funnel": dict(funnel),
    "counts": {s: int((out.split == s).sum()) for s in ["train", "val", "test"]},
    "cities": sorted(out.city.unique().tolist()),
    "label_validation": "pending (E1: human annotation not yet done)",
}
with fs.open(path(OUT_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

funnel_en = ["merged cells", "aerial tile usable", f"at least {MIN_LABELED_DIMS} labeled dimension(s)",
             f"aerial vs street photo year gap <= {MAX_YEAR_GAP}", "OSM features present",
             f"buffer: dropped train/val cells within {BUFFER_RINGS} ring(s) of a test cell"]

def md_table(d):
    return "| " + " | ".join(d.columns) + " |\n|" + "---|" * len(d.columns) + "\n" + "\n".join(
        "| " + " | ".join(str(v) for v in r) + " |" for r in d.itertuples(index=False))

card = f"""# GreenPath dataset {DATASET_VERSION}

Created {manifest['created_at']} · seed {SEED}

## What this is
One row per H3 cell (resolution {out.h3_index.map(h3.get_resolution).iloc[0]}, about 130 m wide). Inputs **X**: a {int(out.tile_px.iloc[0])}x{int(out.tile_px.iloc[0])} NAIP aerial tile covering {out.ground_m.iloc[0]:.0f} m (`tile_path`) and {len(tab_cols)} OpenStreetMap / ESA WorldCover features. Targets **Y**: eight ground-level "ride experience" scores in 0-1 (NaN = missing), produced by a vision-language model from street-level photos. Street photos are used only to create Y; they are not model inputs.

## Size and splits
{md_table(summary.rename(columns={"格子数": "cells", "区块数": "blocks", "占比": "share"}).reset_index())}

Cities: {', '.join(manifest['cities'])}. Splits are assigned by whole H3 blocks (resolution {BLOCK_RES}); cells within {BUFFER_RINGS} ring(s) of a test cell were dropped from train/val. `fold` (0-{N_FOLDS - 1}) assigns train+val blocks to cross-validation folds.

## Filtering
{md_table(pd.DataFrame(list(zip(funnel_en, [n for _, n in funnel])), columns=["step", "cells left"]))}

## Labels
Model `{labels_meta['model']}`, prompt `{labels_meta['prompt_version']}` (sha {labels_meta.get('prompt_sha')}), temperature 0. Per cell: mean of usable photo scores, scaled (score-1)/4. Cells with fewer than the minimum number of usable photos have NaN labels. **Labels are VLM-generated and not yet validated against human ratings (experiment E1).**

Dimensions: {', '.join(DIMS)}.

## Sources and licenses
- Street-level imagery: Mapillary contributors, CC-BY-SA 4.0 (labels only; photos not redistributed here).
- Aerial imagery: USDA NAIP via Microsoft Planetary Computer (public domain).
- Map features: OpenStreetMap contributors, ODbL (Geofabrik snapshot {osm_meta.get('last_modified', 'unknown')}).
- Land cover: ESA WorldCover 2021 v200, CC BY 4.0.

## Known limitations
- Single city unless `test_cities` is set; the test split is held-out blocks, not a held-out city.
- Street photos are crowd-sourced, partly taken from inside vehicles, and concentrated on main roads.
- Street photo dates and aerial year can differ by up to {MAX_YEAR_GAP} years.
"""
with fs.open(path(OUT_DIR, "DATASET_CARD.md"), "w") as f:
    f.write(card)
print(json.dumps(manifest["funnel"], ensure_ascii=False, indent=2))

## 10. 运行记录

In [ ]:
write_manifest({"step": "05_build_dataset", "dataset_version": DATASET_VERSION, "counts": manifest["counts"],
                "funnel": manifest["funnel"], "outputs": [OUT_DIR], "finished_at": now_iso()},
               "05_build_dataset_manifest.json")
print("完成：", OUT_DIR)

## 下一步
- 样本量小（几十到几百个格子）时，train / val / test 都很小，**这一版只用来验证流程**。把 01–04 放大到全量后，升一个版本号重新生成。
- 下一个 notebook：`06_baselines`，在这份数据上训练 B0（均值）、B1（OSM + LightGBM）、M1（冻结编码器 + 线性回归头）、M3（融合）。
- 数据版本管理：把 `datasets/{版本}/` 交给 DVC 追踪，`manifest.json` 里的哈希可以用来核对输入有没有变。